In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [8]:
df = pd.read_csv('../data/tourism_budget.csv')
print (df.shape)
df.head(20)

(100, 11)


,destination,region,accessibility,popularity_score,ideal_days,budget_daily_min,budget_daily_max,mid_daily_min,mid_daily_max,luxury_daily_min,luxury_daily_max
0,Goa,West India,Easy,9,5.0,1400,2800,3300,6600,10800,26100
1,Ladakh (Leh),North India,Moderate,9,7.0,1500,3400,3900,7700,10700,22500
2,Jaipur,North India,Easy,9,3.0,1300,2600,2500,5600,8300,23700
3,Varanasi (Kashi),North India,Easy,9,3.0,1200,2400,2450,5000,6900,23800
4,Agra,North India,Easy,10,2.0,1400,2800,2800,5900,7000,24800
5,Udaipur,West India,Moderate,8,3.0,1400,2700,2550,5700,8000,25300
6,Kerala Backwaters (Alappuzha & Kumarakom),South India,Moderate,9,2.0,1550,3200,3200,6800,8200,25300
7,Coorg (Kodagu),South India,Moderate,7,3.0,1450,2800,2600,5200,7000,19400
8,Ziro Valley,Northeast India,Difficult,5,4.0,1450,2800,2600,5000,4900,11200
9,Mawlynnong & nearby (including Living Root Bri...,Northeast India,Moderate,6,2.0,1450,2800,2600,5000,4900,11200


In [5]:
# Check regions available
print("Regions:", df['region'].unique())
print("\nAccessibility types:", df['accessibility'].unique())
print("\nPopularity range:", df['popularity_score'].min(), "-", df['popularity_score'].max())
print("\nBudget range per day:", df['budget_daily_min'].min(), "-", df['budget_daily_max'].max())
print("\nMissing values:\n", df.isnull().sum())


Regions: <StringArray>
[      'West India',      'North India',      'South India',
  'Northeast India',    'Central India', 'Island Territory',
       'East India', 'North East India',      'Multi-state']
Length: 9, dtype: str

Accessibility types: <StringArray>
['Easy', 'Moderate', 'Difficult']
Length: 3, dtype: str

Popularity range: 4 - 10

Budget range per day: 700 - 12000

Missing values:
 destination         0
region              0
accessibility       0
popularity_score    0
ideal_days          0
budget_daily_min    0
budget_daily_max    0
mid_daily_min       0
mid_daily_max       0
luxury_daily_min    0
luxury_daily_max    0
dtype: int64


In [6]:
# Average daily cost by region
print("Average budget per day by region:")
print(df.groupby('region')[['budget_daily_min', 'mid_daily_min', 'luxury_daily_min']].mean().round(0))

print("\nAverage budget per day by accessibility:")
print(df.groupby('accessibility')[['budget_daily_min', 'mid_daily_min', 'luxury_daily_min']].mean().round(0))

Average budget per day by region:
                  budget_daily_min  mid_daily_min  luxury_daily_min
region                                                             
Central India               2217.0         4300.0            7783.0
East India                  1517.0         3050.0            6033.0
Island Territory            3762.0         7062.0           12700.0
Multi-state                 1200.0         2600.0            5400.0
North East India            2233.0         4292.0            7867.0
North India                 1689.0         3208.0            6706.0
Northeast India             1462.0         2650.0            5038.0
South India                 1390.0         2704.0            5629.0
West India                  1854.0         3558.0            7415.0

Average budget per day by accessibility:
               budget_daily_min  mid_daily_min  luxury_daily_min
accessibility                                                   
Difficult                2619.0         4785.0

In [7]:
df['region']

0        West India
1       North India
2       North India
3       North India
4       North India
          ...      
95      North India
96      North India
97      South India
98    Central India
99      North India
Name: region, Length: 100, dtype: str

In [9]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

# Encode categorical columns
region_enc = LabelEncoder()
access_enc = LabelEncoder()

df['region_encoded'] = region_enc.fit_transform(df['region'])
df['access_encoded'] = access_enc.fit_transform(df['accessibility'])

# Features and targets
X = df[['popularity_score', 'ideal_days', 'region_encoded', 'access_encoded']]

# Train separate models for each budget tier
targets = ['budget_daily_min', 'budget_daily_max', 'mid_daily_min', 'mid_daily_max', 'luxury_daily_min', 'luxury_daily_max']

models = {}
for target in targets:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    models[target] = model
    print(f"{target}: R2={r2:.2f}, MAE=₹{mae:.0f}")

budget_daily_min: R2=0.07, MAE=₹584
budget_daily_max: R2=0.11, MAE=₹1074
mid_daily_min: R2=0.11, MAE=₹1012
mid_daily_max: R2=0.15, MAE=₹1850
luxury_daily_min: R2=0.16, MAE=₹1633
luxury_daily_max: R2=0.29, MAE=₹4188


In [10]:
from sklearn.ensemble import RandomForestRegressor

rf_models = {}
for target in targets:
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rf_models[target] = model
    print(f"{target}: R2={r2:.2f}, MAE=₹{mae:.0f}")

budget_daily_min: R2=-0.09, MAE=₹547
budget_daily_max: R2=-0.66, MAE=₹1240
mid_daily_min: R2=-0.42, MAE=₹1058
mid_daily_max: R2=-0.44, MAE=₹1958
luxury_daily_min: R2=-0.25, MAE=₹1970
luxury_daily_max: R2=0.18, MAE=₹4293


In [11]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score

rf_models = {}
for target in targets:
    y = df[target]
    model = RandomForestRegressor(n_estimators=100, random_state=42)
    
    # Cross validation - more reliable with small datasets
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    print(f"{target}: CV R2 = {cv_scores.mean():.2f} (+/- {cv_scores.std():.2f})")
    
    # Train on ALL data for final model
    model.fit(X, y)
    rf_models[target] = model

budget_daily_min: CV R2 = -0.51 (+/- 0.96)
budget_daily_max: CV R2 = -0.70 (+/- 1.18)
mid_daily_min: CV R2 = -0.66 (+/- 1.15)
mid_daily_max: CV R2 = -0.67 (+/- 1.19)
luxury_daily_min: CV R2 = -0.17 (+/- 0.74)
luxury_daily_max: CV R2 = 0.09 (+/- 0.26)


In [12]:
# Region-based budget lookup (more reliable than ML here)
region_budgets = df.groupby('region')[['budget_daily_min', 'budget_daily_max', 
                                        'mid_daily_min', 'mid_daily_max',
                                        'luxury_daily_min', 'luxury_daily_max']].mean().round(0)

print(region_budgets)

# East India = Jharkhand
print("\nJharkhand (East India) estimates:")
print(region_budgets.loc['East India'])

                  budget_daily_min  budget_daily_max  mid_daily_min  \
region                                                                
Central India               2217.0            4633.0         4300.0   
East India                  1517.0            3567.0         3050.0   
Island Territory            3762.0            8212.0         7062.0   
Multi-state                 1200.0            3150.0         2600.0   
North East India            2233.0            4825.0         4292.0   
North India                 1689.0            3577.0         3208.0   
Northeast India             1462.0            2925.0         2650.0   
South India                 1390.0            3044.0         2704.0   
West India                  1854.0            3869.0         3558.0   

                  mid_daily_max  luxury_daily_min  luxury_daily_max  
region                                                               
Central India            8567.0            7783.0           19550.0  
East Ind